# Data

Graphs $G = (V, E)$ are represented by a set of vertices or nodes $v \in V$ and edges or bonds $e_{v,w} = (v, w) \in E$ between them. For machine learning (ML), vertices and edges are attributed with feature information.

The `kgcnn_torch` package provides data utilities that work with the [PyTorch Geometric (PyG)](https://pyg.org/) ecosystem. Graphs are stored as dictionaries of numpy arrays (following the same conventions as the Keras `kgcnn` package), then converted to PyG `Data` objects for training with PyG `DataLoader`.

## Graph Representation with NetworkX

For visualization and manipulation, [NetworkX](https://networkx.org/) provides convenient graph types. Here is a quick example.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Graph information as arrays.
node_number = [1, 2, 3, 4, 5]
node_attributes = ["A", "B", "C", "D", "E"]
edge_indices = [(1, 2), (1, 3), (1, 5), (2, 3), (3, 4), (4, 5)]

# Setup graph.
G = nx.DiGraph()
G.add_nodes_from(node_number)
G.add_edges_from(edge_indices)
pos = nx.spring_layout(G, seed=42)
options = {
    "font_size": 18, "node_size": 1500, "node_color": "white",
    "edgecolors": "black", "linewidths": 2, "width": 2,
}
nx.draw(G, pos, labels={i: "%s$_{(%s)}$" % (x, i) for i, x in zip(node_number, node_attributes)},
        with_labels=True, **options)
nx.draw_networkx_edge_labels(G, pos, edge_labels={x: x for x in G.edges()}, font_size=18)
plt.show()

## Graph Data as Numpy Dictionaries

In `kgcnn_torch`, graph data is stored as simple Python dictionaries of numpy arrays (compatible with `kgcnn.data.base.GraphDict` from the Keras version). A graph is a collection of:

- `node_attributes`: Node features of shape `(N, F)` where N is the number of nodes and F is the node feature dimension.
- `edge_indices`: Connection list of shape `(M, 2)` where M is the number of edges. The indices denote a connection of incoming (receiving) node `i` and outgoing (sending) node `j` as `(i, j)`.
- `edge_attributes`: Edge features of shape `(M, F)`.
- `graph_attributes`: Graph state information of shape `(F,)`.

Additional properties like labels, positions/coordinates, forces, etc. can be added.

In [ ]:
import numpy as np

# A single graph stored as a dictionary of numpy arrays.
graph = {
    "edge_indices": np.array([[1, 0], [0, 1]]),
    "node_label": np.array([[0], [1]]),
    "graph_labels": np.array([0]),
    "edge_attributes": np.array([[1.0], [2.0]]),
}
print({key: value.shape for key, value in graph.items()})

## Converting to PyG Data Objects

For use with PyTorch Geometric models, we convert graph dictionaries to `torch_geometric.data.Data` objects. This is the standard format expected by all `kgcnn_torch` models.

In [ ]:
import torch
from torch_geometric.data import Data

def graph_dict_to_pyg(g: dict) -> Data:
    """Convert a numpy graph dictionary to a PyG Data object.
    
    This is the fundamental conversion used throughout kgcnn_torch.
    The PyG convention uses:
      - data.x or data.z: node features
      - data.edge_index: (2, M) edge indices (source, target)
      - data.edge_attr: edge features
      - data.y: graph labels
      - data.pos: node positions (for geometric models)
    """
    data_dict = {}
    
    # Edge indices: convert from (M, 2) kgcnn format to (2, M) PyG format
    if "edge_indices" in g:
        ei = np.array(g["edge_indices"])
        data_dict["edge_index"] = torch.tensor(ei.T, dtype=torch.long)
    
    # Node features
    if "node_attributes" in g:
        data_dict["x"] = torch.tensor(g["node_attributes"], dtype=torch.float32)
    if "node_number" in g:
        data_dict["z"] = torch.tensor(g["node_number"], dtype=torch.long)
    
    # Edge features
    if "edge_attributes" in g:
        data_dict["edge_attr"] = torch.tensor(g["edge_attributes"], dtype=torch.float32)
    
    # Graph labels
    if "graph_labels" in g:
        data_dict["y"] = torch.tensor(g["graph_labels"], dtype=torch.float32)
    
    # Node coordinates (for SchNet, PAiNN, etc.)
    if "node_coordinates" in g:
        data_dict["pos"] = torch.tensor(g["node_coordinates"], dtype=torch.float32)
    
    return Data(**data_dict)


# Example: convert our graph dictionary to PyG
pyg_data = graph_dict_to_pyg(graph)
print(pyg_data)
print("Edge index shape:", pyg_data.edge_index.shape)
print("Edge index (PyG convention, 2 x M):\n", pyg_data.edge_index)

## Working with Lists of Graphs

For supervised learning tasks, we work with lists of graph dictionaries. These can be converted to lists of PyG `Data` objects and loaded with PyG's `DataLoader` for batched training.

In [ ]:
# A list of graph dictionaries (like MemoryGraphList in kgcnn Keras)
graph_list = [
    {
        "edge_indices": np.array([[0, 1], [1, 0]]),
        "node_attributes": np.array([[1.0], [2.0]]),
        "graph_labels": np.array([0.5]),
    },
    {
        "edge_indices": np.array([[0, 1], [1, 0], [1, 2], [2, 1]]),
        "node_attributes": np.array([[1.0], [2.0], [3.0]]),
        "graph_labels": np.array([1.2]),
    },
    {
        "edge_indices": np.array([[0, 0]]),
        "node_attributes": np.array([[4.0]]),
        "graph_labels": np.array([0.1]),
    },
]

# Convert to PyG Data list
pyg_list = [graph_dict_to_pyg(g) for g in graph_list]
print("Number of graphs:", len(pyg_list))
for i, d in enumerate(pyg_list):
    print(f"  Graph {i}: {d.num_nodes} nodes, {d.num_edges} edges, label={d.y.item():.1f}")

## Using PyG DataLoader

The PyG `DataLoader` automatically batches graphs into a single disjoint graph with `batch` indices, which is exactly what `kgcnn_torch` models expect.

In [ ]:
from torch_geometric.loader import DataLoader

loader = DataLoader(pyg_list, batch_size=2, shuffle=False)

for batch in loader:
    print("Batch object:", batch)
    print("  x shape:", batch.x.shape)           # All nodes concatenated
    print("  edge_index shape:", batch.edge_index.shape)  # All edges, indices offset per graph
    print("  batch:", batch.batch)                # Graph assignment per node
    print("  y:", batch.y)                        # Labels
    print()

## Using kgcnn_torch Data Utilities

The `kgcnn_torch.data.utils` module provides helper functions for data processing.

In [ ]:
from kgcnn_torch.data.utils import (
    train_test_indices,
    kfold_indices,
    pad_np_array_list_batch_dim,
    list_to_ragged_values,
)

# Train/test split
n_samples = 100
train_idx, test_idx = train_test_indices(n_samples, train_ratio=0.8, seed=42)
print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")

# K-fold cross validation
folds = kfold_indices(n_samples, n_splits=5, seed=42)
for i, (tr, va) in enumerate(folds):
    print(f"  Fold {i}: train={len(tr)}, val={len(va)}")

In [ ]:
# Padding ragged arrays for batched operations
ragged_nodes = [
    np.array([[1.0, 2.0], [3.0, 4.0]]),  # 2 nodes
    np.array([[5.0, 6.0]]),                # 1 node
    np.array([[7.0, 8.0], [9.0, 10.0], [11.0, 12.0]]),  # 3 nodes
]

padded, mask = pad_np_array_list_batch_dim(ragged_nodes)
print("Padded shape:", padded.shape)  # (3, 3, 2) - batch x max_nodes x features
print("Mask shape:", mask.shape)
print("Padded:\n", padded)
print("Mask:\n", mask)

In [ ]:
# Convert list to flat (ragged) representation with row splits
values, row_splits = list_to_ragged_values(ragged_nodes)
print("Flat values shape:", values.shape)   # (6, 2)
print("Row splits:", row_splits)             # [0, 2, 3, 6]

## Using kgcnn_torch DataLoader Utilities

The `kgcnn_torch.io.loader` module provides convenience wrappers for creating PyG DataLoaders.

In [ ]:
from kgcnn_torch.io.loader import get_dataloader, get_train_val_test_loaders

# Simple loader from a list of PyG Data objects
loader = get_dataloader(pyg_list, batch_size=2, shuffle=True)
for batch in loader:
    print("Batch:", batch)

# Train/val split loaders
train_loader, val_loader = get_train_val_test_loaders(
    pyg_list, 
    train_indices=[0, 1], 
    val_indices=[2],
    batch_size=2,
)
print("\nTrain batches:", len(train_loader))
print("Val batches:", len(val_loader))

## Datasets from kgcnn

The `kgcnn_torch` package shares the same data infrastructure as the Keras `kgcnn` package. You can use `kgcnn.data.datasets` to load pre-defined datasets (e.g. `ESOLDataset`, `QM9Dataset`, `MUTAGDataset`) and then convert them to PyG `Data` objects.

> **NOTE**: Loading datasets requires the `kgcnn` Keras package to be installed (for the dataset download/processing infrastructure). The graph data is then converted to PyG format for use with `kgcnn_torch` models.

Below is an example of how to load ESOL and convert to PyG format. This cell requires `kgcnn` (Keras) to be installed and will download data on first run.

In [ ]:
# NOTE: This cell requires kgcnn (Keras) to be installed for dataset loading.
# pip install kgcnn

# from kgcnn.data.datasets.ESOLDataset import ESOLDataset
# 
# dataset = ESOLDataset()
# print("Number of graphs:", len(dataset))
# print("Keys in first graph:", dataset[0].keys())
#
# # Convert each graph dict to a PyG Data object
# pyg_data_list = []
# for g in dataset:
#     data = Data()
#     if "node_attributes" in g:
#         data.x = torch.tensor(np.array(g["node_attributes"]), dtype=torch.float32)
#     if "node_number" in g:
#         data.z = torch.tensor(np.array(g["node_number"]), dtype=torch.long)
#     if "edge_indices" in g:
#         ei = np.array(g["edge_indices"])
#         data.edge_index = torch.tensor(ei.T, dtype=torch.long)
#     if "edge_attributes" in g:
#         data.edge_attr = torch.tensor(np.array(g["edge_attributes"]), dtype=torch.float32)
#     if "graph_labels" in g:
#         data.y = torch.tensor(np.array(g["graph_labels"]), dtype=torch.float32)
#     if "node_coordinates" in g:
#         data.pos = torch.tensor(np.array(g["node_coordinates"]), dtype=torch.float32)
#     pyg_data_list.append(data)
#
# print("Converted", len(pyg_data_list), "graphs to PyG format.")
# print("Example:", pyg_data_list[0])

## Data Scaling

For regression tasks, it is common to standardize targets. Scikit-learn's `StandardScaler` works well, or you can use kgcnn's scalers for molecule-specific scaling.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Example: scale graph labels
labels = np.array([[0.5], [1.2], [0.1], [3.4], [2.1]])
scaler = StandardScaler()
labels_scaled = scaler.fit_transform(labels)

print("Original labels:", labels.ravel())
print("Scaled labels:", labels_scaled.ravel())
print("Inverse:", scaler.inverse_transform(labels_scaled).ravel())

## Complete Example: Data Pipeline for Training

Here is a complete example showing how to set up a data pipeline from raw graph data to PyG DataLoaders ready for training.

In [ ]:
import torch
import numpy as np
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.preprocessing import StandardScaler
from kgcnn_torch.data.utils import train_test_indices

# 1. Create synthetic graph data
np.random.seed(42)
n_graphs = 50
pyg_dataset = []
for i in range(n_graphs):
    n_nodes = np.random.randint(3, 10)
    n_edges = np.random.randint(n_nodes, n_nodes * 3)
    edge_index = torch.randint(0, n_nodes, (2, n_edges))
    x = torch.randn(n_nodes, 4)  # 4-dim node features
    y = torch.tensor([float(n_nodes) + np.random.randn() * 0.1])  # Regression target
    pyg_dataset.append(Data(x=x, edge_index=edge_index, y=y))

# 2. Split into train/test
train_idx, test_idx = train_test_indices(n_graphs, train_ratio=0.8, seed=42)
train_data = [pyg_dataset[i] for i in train_idx]
test_data = [pyg_dataset[i] for i in test_idx]

# 3. Scale labels
train_labels = np.array([d.y.numpy() for d in train_data])
scaler = StandardScaler()
train_labels_scaled = scaler.fit_transform(train_labels)
for d, label in zip(train_data, train_labels_scaled):
    d.y = torch.tensor(label, dtype=torch.float32)

test_labels = np.array([d.y.numpy() for d in test_data])
test_labels_scaled = scaler.transform(test_labels)
for d, label in zip(test_data, test_labels_scaled):
    d.y = torch.tensor(label, dtype=torch.float32)

# 4. Create DataLoaders
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=16, shuffle=False)

print(f"Train: {len(train_data)} graphs, Test: {len(test_data)} graphs")
for batch in train_loader:
    print(f"Batch: {batch.num_graphs} graphs, {batch.x.shape[0]} total nodes")
    break

> **NOTE**: You can find this page as a Jupyter notebook in the `docs/source` directory of the kgcnn-torch repository.